In [1]:

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

# 1. Load dataset
df = pd.read_csv("actual_cleaned_outlier_removed.csv")

# 2. Define the nine predictors
FEATURES = [
    "LOC",
    "nesting_depth",
    "loops",
    "conditions",
    "functions",
    "variables",
    "cyclomatic_complexity",
    "experience_level",
    "python_comfort_1to5"
]

TARGET = "difficulty_1to10"

# 3. Check required columns
required = FEATURES + [TARGET, "snippet_id", "participant_code"]
missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# Keep rows with valid target and grouping IDs
df = df.dropna(subset=[TARGET, "snippet_id", "participant_code"]).copy()

X = df[FEATURES]
y = df[TARGET]

print("Dataset shape:", df.shape)
print("Number of students:", df["participant_code"].nunique())
print("Number of snippets:", df["snippet_id"].nunique())

# 4. Define the Random Forest model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# 5. Evaluate on completely unseen snippets
logo = LeaveOneGroupOut()
groups = df["snippet_id"]

oof_predictions = np.full(len(df), np.nan)

for train_idx, test_idx in logo.split(X, y, groups=groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Fit using training snippets only
    model.fit(X_train, y_train)

    # Predict ratings for the held-out snippet
    oof_predictions[test_idx] = model.predict(X_test)

# 6. Calculate overall out-of-fold metrics
valid = ~np.isnan(oof_predictions)

mae = mean_absolute_error(y[valid], oof_predictions[valid])
rmse = np.sqrt(mean_squared_error(y[valid], oof_predictions[valid]))
r2 = r2_score(y[valid], oof_predictions[valid])

print("\nRandom Forest — unseen-snippet evaluation")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

# 7. Train final model on all available rows
model.fit(X, y)

# 8. Save the trained model
import joblib
joblib.dump(model, "random_forest_difficulty.pkl")

print("\nModel saved as random_forest_difficulty.pkl")

# 9. Show feature importance
importance = pd.Series(
    model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print("\nFeature importance:")
print(importance)


Dataset shape: (1027, 15)
Number of students: 80
Number of snippets: 13

Random Forest — unseen-snippet evaluation
MAE:  1.786
RMSE: 2.305
R²:   0.287

Model saved as random_forest_difficulty.pkl

Feature importance:
python_comfort_1to5      0.357753
variables                0.278695
experience_level         0.223512
LOC                      0.041156
loops                    0.028879
nesting_depth            0.024085
conditions               0.023469
cyclomatic_complexity    0.020213
functions                0.002239
dtype: float64
